# Purchasing star linked to sales
Neeraj Bhuma

Run the Lesson 1 setup/load cells first, in the same Python session, or run them below. Bronze stays raw; Silver applies types and the slide's cleaning rule; Gold contains one row per order line.


In [ ]:
# Exercise 1: set up raw data. No paid resources are created.
import urllib.request, re, json
from pyspark.sql import functions as F
try:
    spark.sql('CREATE CATALOG IF NOT EXISTS adventure_works')
    catalog, lesson_schema = 'adventure_works', 'default'
except Exception as exc:
    print('Catalog creation is unavailable in this workspace:', str(exc)[:400])
    catalog, lesson_schema = 'workspace', 'adventure_works'
lesson = f'{catalog}.{lesson_schema}'
bronze = f'{catalog}.bronze' if catalog == 'adventure_works' else 'workspace.adventure_works_bronze'
silver = f'{catalog}.silver' if catalog == 'adventure_works' else 'workspace.adventure_works_silver'
gold = f'{catalog}.gold' if catalog == 'adventure_works' else 'workspace.adventure_works_gold'
for schema in [lesson, bronze, silver, gold]:
    spark.sql(f'CREATE SCHEMA IF NOT EXISTS {schema}')
spark.sql(f'CREATE VOLUME IF NOT EXISTS {lesson}.raw_data')
raw_dir = f'/Volumes/{catalog}/{lesson_schema}/raw_data'
base = 'https://raw.githubusercontent.com/microsoft/sql-server-samples/master/samples/databases/adventure-works/oltp-install-script'
sources = {'SalesOrderDetail':'Sales', 'Product':'Production', 'SalesTerritory':'Sales', 'SalesOrderHeader':'Sales', 'PurchaseOrderDetail':'Purchasing', 'PurchaseOrderHeader':'Purchasing', 'Vendor':'Purchasing'}
for name in list(sources) + ['instawdb']:
    suffix = '.sql' if name == 'instawdb' else '.csv'
    urllib.request.urlretrieve(base+'/'+name+suffix, raw_dir+'/'+name+suffix)
script = open(raw_dir+'/instawdb.sql').read()
# Read column names from CREATE TABLE, including computed columns in the CSV.
columns = {}
for name, schema in sources.items():
    start = script.index(f'CREATE TABLE [{schema}].[{name}]')
    block = script[start:script.index(') ON [PRIMARY]', start)]
    columns[name] = re.findall(r'^\s*\[([^\]]+)\]', block, re.M)
print('Lesson tables:', lesson, 'Bronze:', bronze, 'Silver:', silver, 'Gold:', gold)
display(dbutils.fs.ls(raw_dir))


In [ ]:
# Exercise 2: apply the source names and types, then save the sales star.
# The CREATE TABLE script is kept alongside the CSVs for checking the mapping.
def raw(name):
    return spark.read.option('sep','\t').option('header',False).csv(raw_dir+'/'+name+'.csv')
def typed(name):
    df = raw(name)
    assert len(df.columns) == len(columns[name]), (name,len(df.columns),columns[name])
    df = df.toDF(*columns[name])
    exprs = []
    for c in df.columns:
        typ = 'string'
        if c.endswith('ID') or c in ['OrderQty','RevisionNumber','Status','CreditRating','SafetyStockLevel','ReorderPoint','DaysToManufacture']:
            typ = 'int'
        elif c.endswith('Date'):
            typ = 'timestamp'
        elif c in ['LineTotal','UnitPrice','UnitPriceDiscount','SubTotal','TaxAmt','Freight','TotalDue','StandardCost','ListPrice','SalesYTD','SalesLastYear','CostYTD','CostLastYear']:
            typ = 'decimal(38,6)' if c == 'LineTotal' and name == 'SalesOrderDetail' else 'decimal(19,4)'
        elif c in ['ReceivedQty','RejectedQty','StockedQty','Weight']:
            typ = 'decimal(18,4)'
        elif c.endswith('Flag') or c == 'PreferredVendorStatus':
            typ = 'boolean'
        exprs.append(F.col(c).cast(typ).alias(c))
    return df.select(*exprs)
products, territories = typed('Product'), typed('SalesTerritory')
sales_detail, sales_header = typed('SalesOrderDetail'), typed('SalesOrderHeader')
assert sales_header.count() == sales_header.select('SalesOrderID').distinct().count()
fact_sales = sales_detail.join(sales_header.select('SalesOrderID','OrderDate','TerritoryID'), 'SalesOrderID')
assert fact_sales.count() == sales_detail.count()
for name, df in [('dim_product',products),('dim_sales_territory',territories),('fact_sales',fact_sales)]:
    df.write.mode('overwrite').option('overwriteSchema',True).saveAsTable(lesson+'.'+name)
    print(name, df.count())
spark.sql(f'USE CATALOG {catalog}')
spark.sql(f'USE SCHEMA {lesson_schema}')


In [ ]:
# Homework: keep Bronze unchanged, clean in Silver, join at line grain in Gold.
for name,target in [('PurchaseOrderDetail','stg_purchase_order_detail'),('PurchaseOrderHeader','stg_purchase_order_header'),('Vendor','dim_vendor')]:
    df = raw(name)
    df.write.mode('overwrite').option('overwriteSchema',True).saveAsTable(bronze+'.'+target)
    print('Bronze',target,df.count(),df.columns)
detail = typed('PurchaseOrderDetail')
before = detail.count()
# Only the two exclusions required by the slide. A NULL amount is retained.
detail = detail.where('ProductID IS NOT NULL AND (LineTotal IS NULL OR LineTotal >= 0)')
print('Purchase detail before:',before,'after:',detail.count(),'dropped:',before-detail.count())
header, vendors = typed('PurchaseOrderHeader'), typed('Vendor')
for name,df in [('stg_purchase_order_detail',detail),('stg_purchase_order_header',header),('dim_vendor',vendors),('dim_product',products),('dim_sales_territory',territories)]:
    df.write.mode('overwrite').option('overwriteSchema',True).saveAsTable(silver+'.'+name)
assert header.count() == header.select('PurchaseOrderID').distinct().count()
fact_purchases = detail.join(header.select('PurchaseOrderID','VendorID','OrderDate'),'PurchaseOrderID')
assert fact_purchases.count() == detail.count()
assert fact_purchases.select('PurchaseOrderID','PurchaseOrderDetailID').distinct().count() == fact_purchases.count()
assert fact_purchases.join(products,'ProductID','left_anti').count() == 0
assert fact_purchases.join(vendors,fact_purchases.VendorID==vendors.BusinessEntityID,'left_anti').count() == 0
fact_purchases.write.mode('overwrite').option('overwriteSchema',True).saveAsTable(gold+'.fact_purchases')
fact_sales.write.mode('overwrite').option('overwriteSchema',True).saveAsTable(gold+'.fact_sales')
print('Gold sales:',fact_sales.count(),'Gold purchases:',fact_purchases.count())
# Query 1: quantity-weighted unit prices. Sales LineTotal includes line discounts.
# Aggregate each fact before joining: a raw fact-to-fact join would multiply lines.
q1 = spark.sql(f'''WITH s AS (
SELECT ProductID,SUM(LineTotal)/SUM(OrderQty) AS sales_price FROM {gold}.fact_sales GROUP BY ProductID),
p AS (SELECT ProductID,SUM(LineTotal)/SUM(OrderQty) AS purchase_price FROM {gold}.fact_purchases GROUP BY ProductID)
SELECT s.ProductID,d.Name,s.sales_price,p.purchase_price,s.sales_price-p.purchase_price AS margin_per_unit
FROM s JOIN p ON s.ProductID=p.ProductID JOIN {silver}.dim_product d ON s.ProductID=d.ProductID
ORDER BY margin_per_unit,s.ProductID LIMIT 5''')
display(q1)
# Query 2: include months where only one side exists, treating missing quantities as zero.
q2 = spark.sql(f'''WITH s AS (
SELECT ProductID,date_trunc('month',OrderDate) AS month,SUM(OrderQty) AS sold FROM {gold}.fact_sales GROUP BY ProductID,date_trunc('month',OrderDate)),
p AS (SELECT ProductID,date_trunc('month',OrderDate) AS month,SUM(OrderQty) AS purchased FROM {gold}.fact_purchases GROUP BY ProductID,date_trunc('month',OrderDate)),
m AS (SELECT COALESCE(s.ProductID,p.ProductID) AS ProductID,COALESCE(s.month,p.month) AS month,COALESCE(s.sold,0) AS sold,COALESCE(p.purchased,0) AS purchased FROM s FULL OUTER JOIN p ON s.ProductID=p.ProductID AND s.month=p.month)
SELECT m.*,d.Name,sold-purchased AS quantity_gap FROM m JOIN {silver}.dim_product d ON m.ProductID=d.ProductID WHERE sold>purchased ORDER BY quantity_gap DESC,m.ProductID,m.month''')
print('Product-months with sold > purchased:',q2.count())
display(q2)
print('Purchase quantity means units ordered, not receipts. A monthly gap alone does not prove a stockout; opening stock and lead times are not modeled.')
q1.write.mode('overwrite').saveAsTable(gold+'.query1_lowest_margins')
q2.write.mode('overwrite').saveAsTable(gold+'.query2_monthly_quantity_gaps')
